### Process Density Data

This file contains code for taking the hourly density map and processing it into a state usable for the project by eliminating features and performing group operations. It also increases the weighting for buildings without air conditioning or faculty buildings. The default scaling rate for both is set at 25%, but can be changed via changing the NO_AC_SCALING and FACULTY_SCALING constants.

In [1]:
import pandas as pd 

# Import the unprocessed data
density_data = pd.read_csv('./data/updated_densitymap_hourly_cleaned.csv')
density_data.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
0,2021-03-31,1,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,1,8,0.13
1,2021-03-31,1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,1,12,0.08
2,2021-03-31,1,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,1,6,0.17
3,2021-03-31,1,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,1,9,0.11
4,2021-03-31,1,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,1,5,0.20


In [4]:
# Set scaling rates for buildings without AC and faculty buildings
NO_AC_SCALING = 1.25
FACULTY_SCALING = 1.25

def get_mean_densities(df):
    df = df.copy()

    # Scale up rows for buildings without air condition and primarily-faculty buildings
    df.loc[(df['building_type'] == 'Residential') & ((df['building_desc'] == '33 HARRY AGGANIS WAY') | (df['building_desc'] == '10 BUICK STREET') | (df['building_desc'] == '610 BEACON STREET')), 'density_cnt'] *= NO_AC_SCALING
    df.loc[(df['building_type'] != 'Residential') | (df['building_type'] == 'Academic'), 'density_cnt'] *= FACULTY_SCALING

    # Drop unnecessary columns and null values
    df = df.drop(columns=['building_desc', 'capacity', 'dc_ratio'])
    df = df.dropna()

    # Group all entries for the same locations together
    mean_densities = df.groupby(by=['latitude', 'longitude'])['density_cnt'].mean().reset_index()
    mean_densities = mean_densities.rename(columns={'density_cnt' : 'mean_density'})

    return mean_densities

def get_max_densities(df):
    df = df.copy()
    
    # Scale up rows for buildings without air condition and primarily-faculty buildings
    df.loc[(df['building_type'] == 'Residential') & ((df['building_desc'] == '33 HARRY AGGANIS WAY') | (df['building_desc'] == '10 BUICK STREET') | (df['building_desc'] == '610 BEACON STREET')), 'density_cnt'] *= NO_AC_SCALING
    df.loc[(df['building_type'] != 'Residential') | (df['building_type'] == 'Academic'), 'density_cnt'] *= FACULTY_SCALING

    # Drop unnecessary columns and null values
    df = df.drop(columns=['building_desc', 'capacity', 'dc_ratio'])
    df = df.dropna()

    # Group all entries for the same locations together
    max_densities = df.groupby(by=['latitude', 'longitude'])['density_cnt'].max().reset_index()
    max_densities = max_densities.rename(columns={'density_cnt' : 'max_density'})

    return max_densities

mean_densities = get_mean_densities(density_data)
max_densities = get_max_densities(density_data)


mean_densities.to_csv('./data/mean_density_data.csv', index=False)
max_densities.to_csv('./data/max_density_data.csv', index=False)

# Do another run with removed residential buildings, most users likely have access to sinks, brita filters, etc.
nonresidential_density_data = density_data[density_data.building_type != 'Residential']

nonresidential_mean_densities = get_mean_densities(nonresidential_density_data)
nonresidential_max_densities = get_max_densities(nonresidential_density_data)

nonresidential_mean_densities.to_csv('./data/nonresidential_mean_density_data.csv', index=False)
nonresidential_max_densities.to_csv('./data/nonresidential_max_density_data.csv', index=False)